# WaveGuard — Colab GPU 자동 라벨링 + yolo26s 파인튜닝

한 노트북으로 **정확 라벨링(교사 모델, GPU)** → **train/val 분할** → **yolo26s 학습** → **가중치 내보내기**를 한다.

## 준비
1. 로컬(vision/)에서: `powershell -ExecutionPolicy Bypass -File .\finetune\pack_for_colab.ps1`
2. 생성된 `vision/finetune/gwangalli_colab.zip` 를 **Google Drive 최상위(MyDrive)** 에 업로드
3. Colab 메뉴 **런타임 > 런타임 유형 변경 > T4 GPU** 선택 후 이 노트북을 위에서부터 실행

> 모델 가중치(yolo26m/yolo26s)는 ultralytics가 자동 다운로드한다. 라벨은 수집 프레임으로부터 GPU가 만든다.

## 1. GPU 확인

In [ ]:
!nvidia-smi

## 2. 패키지 설치 (ultralytics + sahi)

In [ ]:
!pip install -q ultralytics sahi openai-clip
import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 3. Google Drive 마운트 + 패키지 압축 해제
업로드한 `gwangalli_colab.zip` 경로를 `PKG` 에 맞게 수정.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, zipfile, shutil
PKG = '/content/drive/MyDrive/gwangalli_colab.zip'   # ← 업로드한 zip 경로
assert os.path.exists(PKG), f'zip 없음: {PKG} (Drive 경로 확인)'
if os.path.exists('/content/pkg'): shutil.rmtree('/content/pkg')
with zipfile.ZipFile(PKG) as z: z.extractall('/content/pkg')
%cd /content/pkg/vision
n = len([f for f in os.listdir('finetune/raw') if f.endswith('.jpg')])
print('raw frames:', n)

## 4. GPU 자동 라벨링 (교사 모델 = yolo26m + 원근 밴드 SAHI)
GPU라 수백 장도 몇 분이면 끝난다. 결과: `finetune/dataset/{images,labels,preview}/`.
검수: 아래 셀로 preview 몇 장을 보고 명확히 이상한 프레임만 삭제하면 정확도가 더 오른다(선택).

In [ ]:
!python finetune/prelabel.py --teacher

## 4.2 튜브 자동 라벨 (class 1)

피서객은 물에서만 튜브를 쓰므로 **튜브 = 사람 1명 지표**다. COCO에는 tube 클래스가 없어, 오픈보캐뷸러리 모델(YOLO-World)로 물 구역의 튜브를 자동 라벨링해 class 1로 추가한다. 학습 후 서버가 튜브를 사람 1명으로 집계한다(사람 박스와 겹치면 중복 제거).

In [ ]:
# 튜브 자동 라벨: YOLO-World (GPU). 기존 person 라벨 뒤에 class 1 줄을 append.
from ultralytics import YOLOWorld
import cv2, glob, os

tm = YOLOWorld('yolov8l-worldv2.pt')
tm.set_classes(['swim ring', 'swim tube', 'inflatable ring', 'pool float'])

WATER_TOP, WATER_BOT = 0.45, 0.78   # 물 구역만 (모래에 방치된 튜브 제외)
img_dir = 'finetune/dataset/images'
lbl_dir = 'finetune/dataset/labels'
prev_dir = 'finetune/dataset/preview'
n_img = n_tube = 0
for ip in sorted(glob.glob(img_dir + '/*.jpg')):
    stem = os.path.splitext(os.path.basename(ip))[0]
    img = cv2.imread(ip)
    if img is None:
        continue
    h, w = img.shape[:2]
    r = tm.predict(img, conf=0.2, imgsz=1280, verbose=False, device=0)[0]
    lines = []
    vis_path = f'{prev_dir}/{stem}.jpg'
    vis = cv2.imread(vis_path)
    for b in (r.boxes or []):
        x1, y1, x2, y2 = b.xyxy[0].tolist()
        cy = ((y1 + y2) / 2) / h
        if not (WATER_TOP <= cy <= WATER_BOT):
            continue
        cx = ((x1 + x2) / 2) / w
        lines.append(f"1 {cx:.6f} {cy:.6f} {(x2 - x1) / w:.6f} {(y2 - y1) / h:.6f}")
        if vis is not None:
            cv2.rectangle(vis, (int(x1), int(y1)), (int(x2), int(y2)), (255, 200, 0), 2)
    if lines:
        lp = f'{lbl_dir}/{stem}.txt'
        old = open(lp, encoding='utf-8').read().strip() if os.path.exists(lp) else ''
        body = (old + '\n' if old else '') + '\n'.join(lines)
        open(lp, 'w', encoding='utf-8').write(body)
        if vis is not None:
            cv2.imwrite(vis_path, vis)
        n_img += 1
        n_tube += len(lines)
print(f'tube 라벨 {n_tube}개 추가 ({n_img}장, preview에 하늘색 박스)')

In [ ]:
# (선택) 라벨 품질 육안 검수 — preview 몇 장 미리보기
import glob, random
from IPython.display import Image, display
prev = sorted(glob.glob('finetune/dataset/preview/*.jpg'))
print('preview:', len(prev))
for p in random.sample(prev, min(3, len(prev))):
    print(p); display(Image(p, width=720))

## 4.5 의심 프레임 검수 (선택)

자동 라벨 중 '파도 오탐'처럼 이상한 프레임을 골라 `finetune/review/`에 모으고, 아래에서 한 화면에 모두 보여준다. 나쁜 프레임의 파일명을 다음 셀 `bad` 목록에 적어 실행하면 데이터셋에서 제거된다. (검수를 건너뛰려면 이 절 전체를 실행하지 않으면 된다.)

In [ ]:
!python finetune/flag_suspect.py
# 의심 프레임을 한 화면에 모두 표시 (박스 그려진 preview)
import glob
from IPython.display import Image, display
sus = sorted(glob.glob('finetune/review/*.jpg'))
print('의심 프레임:', len(sus))
for p in sus:
    print(p)
    display(Image(p, width=640))

In [ ]:
# 위에서 본 '나쁜' 프레임 파일명(확장자 없이)을 여기에 적고 실행 → 데이터셋에서 제거
bad = [
    # 'gwangalli_20260730_110821',
    # 'gwangalli_20260730_111037',
]
import os
n = 0
for stem in bad:
    for p in (f'finetune/dataset/images/{stem}.jpg',
              f'finetune/dataset/labels/{stem}.txt',
              f'finetune/dataset/preview/{stem}.jpg',
              f'finetune/review/{stem}.jpg'):
        if os.path.exists(p):
            os.remove(p)
    n += 1
    print('removed', stem)
print(f'제거 {n}장. 이제 아래 5절(make_dataset)부터 이어서 실행하세요.')

## 5. train/val 분할 + data.yaml (절대경로)

In [ ]:
!python finetune/make_dataset.py --val 0.2
# Colab 절대경로로 data.yaml 재작성 (ultralytics 경로 혼선 방지)
root = '/content/pkg/vision/finetune/dataset'
open(root+'/data.yaml','w').write(f'path: {root}\ntrain: train/images\nval: val/images\nnames:\n  0: person\n  1: tube\n')
print(open(root+'/data.yaml').read())

## 6. yolo26s 파인튜닝 (GPU)
메모리 부족 시 `batch` 를 낮추거나 `imgsz` 를 768로.

In [ ]:
!yolo detect train model=yolo26s.pt data=/content/pkg/vision/finetune/dataset/data.yaml \
  epochs=100 imgsz=1024 batch=8 device=0 patience=30 \
  project=/content/runs name=gwangalli

## 7. 검증 + 가중치 Drive로 내보내기

In [ ]:
!yolo detect val model=/content/runs/gwangalli/weights/best.pt data=/content/pkg/vision/finetune/dataset/data.yaml device=0

In [ ]:
import shutil, os
src = '/content/runs/gwangalli/weights/best.pt'
dst = '/content/drive/MyDrive/yolo26s_beach_ft.pt'
shutil.copy(src, dst)
print('saved ->', dst, os.path.getsize(dst)//1024, 'KB')

## 8. 로컬 적용
Drive의 `yolo26s_beach_ft.pt` 를 다운로드해서 다음 위치에 넣기:

```
vision/models/yolo26s_beach_ft.pt
```

`realtime_safety_map.py` 가 `FAST_SAHI_MODEL_CANDIDATES` 첫 항목으로 이 파일을 **최우선 자동 로드**한다. 서버를 재기동하면 파인튜닝된 모델로 탐지한다.